# Gerador de images de roupas

## Utilizando Rede GAN

Rede GAN(Rede Adversária Generativa): Essa rede é composta pela rede geradora e por uma rede discriminadora, e ambas precisam ser treinadas em conjunto. Treinar apenas a rede geradora não resulta em boas imagens.

In [ ]:
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import warnings
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import time
from IPython.display import clear_output
from tensorflow.keras.models import load_model
import numpy as np
from PIL import Image
import math
from tensorflow import keras


warnings.filterwarnings('ignore')


In [ ]:
(train_images, train_labels), _ = tf.keras.datasets.fashion_mnist.load_data()

In [ ]:
train_images[0]

In [ ]:
train_images.dtype

Normalizando e adicionando dimensão extra para imagens de treino para quando trabalhar com imagens coloridas

In [ ]:
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1).astype('float32')
train_images = (train_images - 127.5/ 127.5) # Normalizar para [-1, 1]

bath_size = 256

train_ds = tf.data.Dataset.from_tensor_slices(train_images).shuffle(60000).batch(bath_size)
num_images_to_show = 5
plt.figure(figsize=(10,10))
for image_batch in range(num_images_to_show):
  plt.subplot(1, num_images_to_show, image_batch + 1)
  plt.imshow(train_images[image_batch].reshape(28, 28), cmap='gray')
  plt.axis('off')

In [ ]:
def construir_gerador():
  model = Sequential()
  model.add(layers.Input(shape=(100,)))
  model.add(layers.Dense(7*7*256, use_bias=False))
  model.add(layers.BatchNormalization())
  model.add(layers.LeakyReLU())

  model.add(layers.Reshape((7, 7, 256)))
  model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1),
                                   padding='same', use_bias=False))
  model.add(layers.BatchNormalization())
  model.add(layers.LeakyReLU())

  model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2),
                                   padding='same', use_bias=False))
  model.add(layers.BatchNormalization())
  model.add(layers.LeakyReLU())

  model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2),
                                   padding='same', use_bias=False,
                                   activation='tanh'))

  return model

In [ ]:
gerador = construir_gerador()

dimensao_ruido = 100
ruido = tf.random.normal([1, dimensao_ruido])

In [ ]:
img_gerada = gerador(ruido, training=False)
plt.imshow(img_gerada[0]*127.5+127.5)
plt.axis('off');

In [ ]:
def constroi_discriminador():
    model = Sequential()

    model.add(layers.Input(shape=(28, 28, 1)))
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    # Corrigir a dimensão da Flatten para compatibilizar com a Dense
    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

In [ ]:
discriminador = constroi_discriminador()
decisao = discriminador(img_gerada)
print(decisao)

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def custo_discriminador(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def custo_gerador(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

In [ ]:
otimizador_gerador = tf.keras.optimizers.Adam(1e-4)
otimizador_discriminador = tf.keras.optimizers.Adam(1e-4)

In [ ]:
checkpoint_dir = './content/training_checkpoints/'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=otimizador_gerador,
                                 discriminator_optimizer=otimizador_discriminador,
                                 generator=gerador,
                                 discriminator=discriminador)

In [ ]:
epochs = 50
dimensao_ruido = 100
num_exemplos_treinamento = 16

seed = tf.random.normal([num_exemplos_treinamento, dimensao_ruido])

In [ ]:
@tf.function
def passo_treino(imgs):
  ruido = tf.random.normal([bath_size, dimensao_ruido])

  with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
    generated_imgs = gerador(ruido, training=True)

    real_output = discriminador(imgs, training=True)
    fake_output = discriminador(generated_imgs, training=True)

    gen_loss = custo_gerador(fake_output)
    disc_loss = custo_discriminador(real_output, fake_output)

  gradients_of_generator = gen_tape.gradient(gen_loss, gerador.trainable_variables)
  gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminador.trainable_variables)

  otimizador_gerador.apply_gradients(zip(gradients_of_generator, gerador.trainable_variables))
  otimizador_discriminador.apply_gradients(zip(gradients_of_discriminator, discriminador.trainable_variables))


In [ ]:
def train(dataset, epochs):
  for epoch in range(epochs):
    start = time.time()

    for imgs_batch in dataset:
      passo_treino(imgs_batch)

    clear_output(wait=True)
    generate_and_save_images(gerador,
                             epoch + 1,
                             seed)

    if((epoch + 1) % 15 == 0):
      checkpoint.save(file_prefix = checkpoint_prefix)

    print ('Time for epoch {} is {} sec'.format(epoch + 1, time.time()-start))

  clear_output(wait=True)
  generate_and_save_images(gerador,
                           epoch + 1,
                           seed)

In [ ]:
def generate_and_save_images(modelo, epoca, entrada):
  # Observe que `training` está definido como False.
  # Isso é para que todas as camadas sejam executadas no modo de inferência (batchnorm).
  previsao = modelo(entrada, training=False)

  fig = plt.figure(figsize=(4, 4))

  for i in range(previsao.shape[0]):
      plt.subplot(4, 4, i+1)
      plt.imshow(previsao[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
      plt.axis('off')

  plt.savefig('image_at_epoch_{:04d}.png'.format(epoca))
  plt.show()

In [ ]:
train(train_ds, epochs)

In [ ]:
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

gerador.save('modelo_gerador.keras')

In [ ]:
gerador_carregado = load_model('modelo_gerador.keras')

In [ ]:
new_noise = tf.random.normal([1, dimensao_ruido])
new_generated_img = gerador_carregado(new_noise, training=False)
plt.imshow((new_generated_img[0] * 127.5 + 127.5).numpy())
plt.axis('off')
plt.show()

## Utilizando Rede Difusora

A idéia é sair do ruído das GAN's para uma imagem que faça sentido. Vamos passar uma imagem totalmente ruído e ela vai ser filtrada para obter uma imagem perfeita.



In [ ]:
from tqdm.auto import trange, tqdm

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

X_train = (X_train - 127.5/ 1.0) # Normalizar para [-1, 1]

X_train = np.expand_dims(X_train, axis=-1) # Adicionar canal extra para imagens 28x28

IMG_SIZE = 28
BATCH_SIZE = 128
timesteps = 16
time_bar = 1 - np.linspace(0, 1.0, timesteps + 1)

In [ ]:
def cvtImg(img):
  img = img - img.min()
  img = (img/img.max())

  return img.astype(np.float32)

def show_examples(x):
    num_images = x.shape[0]
    plt.figure(figsize=(10, 10))
    for i in range(min(25, num_images)):
        plt.subplot(5, 5, i+1)
        img = cvtImg(x[i])
        plt.imshow(img.squeeze(), cmap='gray')
        plt.axis('off')

In [ ]:
show_examples(X_train)

In [ ]:
def forward_noise(x, t):
    a = time_bar[t]
    b = time_bar[t + 1]

    ruido = np.random.normal(size=x.shape)

    a = a.reshape((-1, 1, 1, 1))
    b = b.reshape((-1, 1, 1, 1))

    img_a = x * (1 - a) + ruido * a
    img_b = x * (1 - b) + ruido * b

    return img_a, img_b

def generate_ts(num):
    return np.random.randint(0, timesteps, size=num)

In [ ]:
t = generate_ts(3)
a, b = forward_noise(X_train[:3], t)
show_examples(a)

In [ ]:
show_examples(b)

In [ ]:
def block(x):
  x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
  x = layers.LayerNormalization()(x)
  x = layers.Activation('relu')(x)

  return x

In [ ]:
def make_model():
  x = x_input = layers.Input(shape=(28, 28, 1), name = 'x_input')
  x_ts = x_ts_input = layers.Input(shape=(1,), name='x_ts_input')
  x_ts = layers.Dense(192)(x_ts)
  x_ts = layers.Activation('relu')(x_ts)

  # ----- left ( down ) -----
  x = x28 = block(x) # redução de dimensionalidade
  x = layers.MaxPool2D(2, padding='same')(x)

  x = x14 = block(x)
  x = layers.MaxPool2D(2, padding='same')(x)

  x = x7 = block(x)
  x = layers.MaxPool2D(2, padding='same')(x)

  x = x4 = block(x)

  # ----- MLP -----
  x = layers.Flatten()(x)
  x = layers.Concatenate()([x, x_ts])
  x = layers.Dense(128)(x)
  x = layers.LayerNormalization()(x)
  x = layers.Activation('relu')(x)

  x = layers.Dense(4 * 4 * 32)(x)
  x = layers.LayerNormalization()(x)
  x = layers.Activation('relu')(x)
  x = layers.Reshape((4, 4, 32))(x)


  # ----- right ( up ) -----
  x = layers.Concatenate()([x, x4])
  x = block(x)
  x = layers.Conv2DTranspose(128, kernel_size=3, strides=2, padding='same')(x)  # 4x4 -> 8x8

  # Ajuste para 7x7
  x = layers.Cropping2D(((0, 1), (0, 1)))(x)  # 8x8 -> 7x7

  x = layers.Concatenate()([x, x7])
  x = block(x)
  x = layers.Conv2DTranspose(128, kernel_size=3, strides=2, padding='same')(x)  # 7x7 -> 14x14

  x = layers.Concatenate()([x, x14])
  x = block(x)
  x = layers.Conv2DTranspose(128, kernel_size=3, strides=2, padding='same')(x)  # 14x14 -> 28x28

  x = layers.Concatenate()([x, x28])
  x = block(x)

  # ----- output -----
  x = layers.Conv2D(1, kernel_size=1, padding='same')(x)
  model = tf.keras.models.Model([x_input, x_ts_input], x)

  return model

In [ ]:
model = make_model()

In [ ]:
model.compile(loss=tf.keras.losses.MeanAbsoluteError(),
              optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0008))

In [ ]:
def predict(model, timesteps=50, batch_size=32):
  x = np.random.normal(size=(batch_size, 28, 28, 1))

  for i in trange(timesteps):
    t = np.full((batch_size, 1), i)
    x = model.predict([x, t], verbose = 0)

  x = (x-x.min())/(x.max()-x.min())

  show_examples(x)

In [ ]:
predict(model)

In [ ]:
def predict_step(model, timesteps=50, num_samples=8):

  xs= []
  x = np.random.normal(size=(num_samples, 28, 28, 1))

  for i in trange(timesteps):
    t = np.full((num_samples, 1), i)
    x = model.predict([x,t] , verbose = 0)
    if i%5 == 0:
      xs.append(x[0])

  xs = [(x - x.min()) / (x.max() - x.min()) for x in xs]

  plt.figure(figsize=(20, 3))
  for i, img in enumerate(xs):
      plt.subplot(1, len(xs), i+1)
      plt.imshow(cvtImg(img), cmap='gray')
      plt.title(f'Step {i*5}')
      plt.axis('off')
  plt.tight_layout()
  plt.show()

In [ ]:
predict_step(model)

In [ ]:
def train_one(x_img):
    x_ts = generate_ts(len(x_img))
    x_a, x_b = forward_noise(x_img, x_ts)
    loss = model.train_on_batch([x_a, x_ts], x_b)
    return loss

def train(R=50):
  bar = trange(R)
  total = 100

  for i in bar:
    for j in range(total):
      x_img = X_train[np.random.randint(len(X_train), size=BATCH_SIZE)]
      loss = train_one(x_img)
      pg = (j/total) * 100
      if j%5 == 0:
        bar.set_description(f'loss: {loss:.5f}, p: {pg:.2f}%')

In [ ]:
train()

In [ ]:
predict(model)

In [ ]:
predict_step(model)

## Modelo pré-treinado Stable Diffusion

In [ ]:
import keras_cv

In [ ]:
# pip install tensorflow==2.15.1 keras==2.15.0 keras-core==0.1.7 keras-cv==0.9.0

In [ ]:
model = keras_cv.models.StableDiffusion(img_width=512, img_height=512)

In [ ]:
images = model.text_to_image(
    "Humanoid cat wearing golden jeans, dark fantasy art, "
    "high quality, highly detailed, elegant, sharp focus, "
    "concept art, character concepts, digital painting, mystery, adventure",
    batch_size=3,
)

In [ ]:
def plot_images(images):
    plt.figure(figsize=(20, 20))
    for i in range(len(images)):
        ax = plt.subplot(1, len(images), i + 1)
        plt.imshow(images[i])
        plt.axis("off")

In [ ]:
plot_images(images)

### Gerando animações com o Stable Difusion

In [ ]:
keras.mixed_precision.set_global_policy("mixed_float16")

In [ ]:
model = keras_cv.models.StableDiffusion(jit_compile=True)

In [ ]:
prompt_1 = "Panda wearing a blue hat, dark fantasy art, "
prompt_2 = "Cat wearing a blue hat, dark fantasy art, "
interpolation_steps = 5

encoding_1 = tf.squeeze(model.encode_text(prompt_1))
encoding_2 = tf.squeeze(model.encode_text(prompt_2))

interpolated_encodings = tf.linspace(encoding_1, encoding_2, interpolation_steps)

# Show the size of the latent manifold
print(f"Encoding shape: {encoding_1.shape}")

In [ ]:
seed = 454
noise = tf.random.normal((512 // 8, 512 // 8, 4), seed=seed)

images = model.generate_image(
    interpolated_encodings,
    batch_size=interpolation_steps,
    diffusion_noise=noise,
)

In [ ]:
def export_as_gif(filename, images, frames_per_second=10, rubber_band=False):
    if rubber_band:
        images += images[2:-1][::-1]
    images[0].save(
        filename,
        save_all=True,
        append_images=images[1:],
        duration=1000 // frames_per_second,
        loop=0,
    )

In [ ]:
export_as_gif(
    "panda-cat.gif",
    [Image.fromarray(img) for img in images],
    frames_per_second=2,
    rubber_band=True,
)

In [ ]:
from IPython.display import Image as IImage
IImage("panda-cat.gif")

Interpolação manual dos resultados

In [ ]:
interpolation_steps = 150
batch_size = 3
batches = interpolation // batch_size

interpolated_encodings = tf.linspace(encoding_1, encoding_2, interpolation_steps)
batched_encodings = tf.split(interpolated_encodings, batches)

images = []

for batch in range(batches):
  images += [
      Image.fromarray(img)
      for img in model.generate_image(
          batched_encodings[batch],
          batch_size=batch_size,
          num_steps = 25,
          diffusion_noise = noise
      )
  ]

  export_as_gif(
    "panda-cat-fino.gif", images, rubber_band= True
)

Criando um caminho circular com ruído

In [ ]:
seed = 454
tf.random.set_seed(seed)

prompt = "A majestic cat wearing an ornate golden hat, surrounded by floating orbs of light, in a dark illuminist painting, high detail, cinematic lighting, surreal background, elegant fur texture"
encoding = tf.squeeze(model.encode_text(prompt))

walk_steps = 150
batch_size = 3
batches = walk_steps // batch_size

noise = tf.random.normal((512 // 8, 512 // 8, 4), seed=seed)  # Ruído inicial fixo

# Gerando ruídos circulares com a mesma seed
walk_noise_x = tf.random.normal(noise.shape, dtype="float64", seed=seed)
walk_noise_y = tf.random.normal(noise.shape, dtype="float64", seed=seed)

# Caminhada circular usando coseno e seno
walk_scale_x = tf.cos(tf.linspace(0, 4, walk_steps) * math.pi)
walk_scale_y = tf.sin(tf.linspace(0, 4, walk_steps) * math.pi)

# Ruído circular aplicado
noise_x = tf.tensordot(walk_scale_x, walk_noise_x, axes=0)
noise_y = tf.tensordot(walk_scale_y, walk_noise_y, axes=0)
noise = tf.add(noise_x, noise_y)

# Dividindo o ruído em lotes
batched_noise = tf.split(noise, batches)

# Gerando imagens sem passar a seed, já que o ruído é manual
images = []
for batch in range(batches):
    images += [
        Image.fromarray(img)
        for img in model.generate_image(
            encoding,
            batch_size=batch_size,
            num_steps=25,
            diffusion_noise=batched_noise[batch],  # Usando apenas o ruído gerado
        )
    ]

# Exportar como GIF com efeito de "vai e volta"
export_as_gif("cat_hat_variation.gif", images, rubber_band=True)

## Usando o modelo do HuggingFace

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import matplotlib.pyplot as plt

pipe = StableDiffusionPipeline.from_pretrained("sd-legacy/stable-diffusion-v1-5", torch_dtype=torch.float16)
pipe.to("cuda")  # Se estiver usando GPU

prompt = "A cute cat using jeans"
image = pipe(prompt).images[0]

plt.imshow(image)
plt.axis("off")
plt.show()


In [ ]:
pipe = StableDiffusionPipeline.from_pretrained("sd-legacy/stable-diffusion-v1-5", torch_dtype=torch.float16)
pipe.to("cuda")  # Se estiver usando GPU

prompt = "microscope image of different and varied microorganisms"

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i in range(3):
    image = pipe(prompt).images[0]

    axes[i].imshow(image)
    axes[i].axis("off")
    axes[i].set_title(f"Captcha {i+1}")

plt.tight_layout()
plt.show()

# Pontos importantes

* Carregar os dados de imagens Fashion MNIST a partir do Keras;
* Realizar transformação das imagens a partir do resize e normalização;
* Configurar a estrutura de camadas de uma rede neural;
* Construir um gerador de imagens a partir de uma rede neural;
* Gerar uma imagem a partir de um ruído.
* Implementar uma Rede Generativa Adversarial (GAN), compreendendo seus dois componentes principais: o gerador e o discriminador;
* Estruturar a rede discriminadora utilizando camadas de convolução, funções de ativação LeakyReLU e Dropout para prevenir overfitting;
* Definir corretamente as funções de custo para o treinamento do gerador e do discriminador, utilizando a função BinaryCrossentropy para calcular as perdas;
* Implementar o loop de treinamento das redes, utilizando GradientTape para calcular os gradientes e aplicar otimização com base nas funções de custo;
* Treinar a GAN ao longo de várias épocas e acompanhar a evolução do modelo gerador;
* Salvar e carregar o modelo gerador para reutilizar e gerar novas imagens a partir de vetores de ruído aleatórios, mesmo após o treinamento.
* Carregar os dados de imagens Fashion MNIST a partir do Keras;
* Normalizar as imagens e adicionar ruído para preparar os dados para a rede difusora;
* Definir a estrutura de uma rede neural em formato de U (U-Net) para a geração de imagens;
* Implementar funções para adicionar diferentes níveis de ruído às imagens;
* Treinar uma U-Net para transformar imagens ruidosas em imagens limpas;
* Criar funções de previsão para visualizar a evolução das imagens geradas a partir do ruído ao longo do tempo.
* Utilizar o modelo Stable Diffusion para gerar imagens a partir de texto.;
* Criar prompts detalhados para gerar imagens com características específicas, como estilo artístico e qualidade.;
* Melhorar a precisão do modelo Stable Diffusion usando a técnica de precisão mista (mixed precision), que reduz o uso de memória e acelera o processo de geração de imagens.;
* Gerar imagens de alta qualidade com detalhes específicos, usando prompts detalhados e ajustando os hiperparâmetros do modelo.;
* Explorar os recursos do Stable Diffusion para gerar imagens com estilos artísticos, cenários e elementos específicos.
* Criar animações com Stable Diffusion, combinando prompts e interpolação de encodings;
* Gerar sequências de imagens mais suaves utilizando interpolação manual em lotes;
* Controlar as variações na imagem usando ruído circular para criar animações com pequenas modificações a partir de um único prompt;
* Usar a função tf.linspace para gerar uma sequência de valores intermediários entre dois prompts;
* Gerar imagens com Stable Diffusion usando um ruído gaussiano pré-definido como ponto de partida para a geração das imagens.